In [ ]:
import json
import numpy as np
import pandas as pd

## データの読み込み

In [ ]:
apt_path = "atp_calculated_results/atp_results_gemma-2-9b-it_20251201_093733.json"
metrics_path = "results/shap_experiments/feature_selection_prompt_last_token_20251129_214443/data/feature_metrics_full.csv"

with open(apt_path, "r") as f:
    data_apt = json.load(f)

data_metrics = pd.read_csv(metrics_path)

In [ ]:
data_metrics.head()

## データ構造の確認


In [ ]:
data_apt["results"][1]['variations'][2]["atp_analysis"]

## atpスコアの計算にエラーがないかのチェック

In [ ]:
# atp_analysisにerrorが含まれているかをチェック
error_count = 0
error_details = []

for idx, result in enumerate(data_apt["results"]):
    question_id = result.get("question_id", idx)
    
    for var_idx, variation in enumerate(result.get("variations", [])):
        # baseテンプレート(template_type=="")はスキップ
        template_type = variation.get("template_type", "")
        if template_type == "":
            continue
        
        # sycophancy_flag==0の場合もスキップ
        sycophancy_flag = variation.get("sycophancy_flag", 0)
        if sycophancy_flag == 0:
            continue
            
        atp_analysis = variation.get("atp_analysis")
        
        # atp_analysisが存在しない場合
        if atp_analysis is None:
            error_count += 1
            error_details.append({
                "question_id": question_id,
                "variation_index": var_idx,
                "error_type": "missing_atp_analysis",
                "template": variation.get("template_type", "unknown")
            })
        # atp_analysisにerrorキーが含まれている場合
        elif isinstance(atp_analysis, dict) and "error" in atp_analysis:
            error_count += 1
            error_details.append({
                "question_id": question_id,
                "variation_index": var_idx,
                "error_type": "error_in_atp_analysis",
                "error_message": atp_analysis.get("error"),
                "template": variation.get("template_type", "unknown")
            })

print(f"Total errors found: {error_count}")
print(f"\nError details:")
for error in error_details:
    print(f"  Question ID: {error['question_id']}, Variation: {error['variation_index']}, "
          f"Template: {error.get('template')}, Type: {error['error_type']}")
    if "error_message" in error:
        print(f"    Error message: {error['error_message']}")

In [ ]:
# 各SAE特徴ごとのATPスコアを集計（Global Mean Attribution方式）

# Step 1: 全迎合サンプル数をカウント
total_sycophancy_samples = 0

for result in data_apt["results"]:
    for variation in result["variations"]:
        # baseテンプレートまたはsycophancy_flag==0の場合はスキップ
        if variation["template_type"] == "" or variation["sycophancy_flag"] == 0:
            continue
        
        atp_analysis = variation["atp_analysis"]
        if atp_analysis is None or "error" in atp_analysis:
            continue
        
        total_sycophancy_samples += 1

print(f"Total sycophancy samples (N_syc): {total_sycophancy_samples}")

# Step 2: 各特徴量のAtPスコア総和を計算（活性化しなかった場合は0として扱う）
feature_atp_sum = {}
feature_activation_count = {}  # 参考用：実際に活性化した回数

for result in data_apt["results"]:
    for variation in result["variations"]:
        # baseテンプレートまたはsycophancy_flag==0の場合はスキップ
        if variation["template_type"] == "" or variation["sycophancy_flag"] == 0:
            continue
        
        atp_analysis = variation["atp_analysis"]
        if atp_analysis is None or "error" in atp_analysis:
            continue
        
        # top_featuresからスコアを取得
        for feature_info in atp_analysis["top_features"]:
            feature_id = feature_info["id"]
            atp_score = feature_info["score"]
            
            if feature_id is not None and atp_score is not None:
                if feature_id not in feature_atp_sum:
                    feature_atp_sum[feature_id] = 0.0
                    feature_activation_count[feature_id] = 0
                
                feature_atp_sum[feature_id] += atp_score
                feature_activation_count[feature_id] += 1

# Step 3: Global Mean Attribution スコアを計算
atp_stats = []
for feature_id, total_score in feature_atp_sum.items():
    # Global Mean: 全迎合サンプル数で割る（非活性時は0として扱う）
    global_mean_score = total_score / total_sycophancy_samples
    
    # 参考値: 活性化した時のみの平均（Conditional Mean）
    conditional_mean_score = total_score / feature_activation_count[feature_id]
    
    atp_stats.append({
        'Feature_ID': feature_id,
        'Global_Mean_ATP': global_mean_score,  # 推奨指標
        'Conditional_Mean_ATP': conditional_mean_score,  # 参考値
        'Total_ATP_Sum': total_score,
        'Activation_Count': feature_activation_count[feature_id],
        'Activation_Rate': feature_activation_count[feature_id] / total_sycophancy_samples
    })

# データフレーム化してGlobal_Mean_ATPでソート
df_atp_scores = pd.DataFrame(atp_stats).sort_values('Global_Mean_ATP', ascending=False).reset_index(drop=True)

print(f"\nTop 10 features by Global Mean ATP Score:")
df_atp_scores.head(10)

## Global Mean vs Conditional Mean の比較

Global Mean（推奨）とConditional Mean（従来方式）の違いを可視化します。
頻度と強度のバランスが正しく評価されているか確認しましょう。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 上位50特徴を抽出して比較
top_n = 50
df_top = df_atp_scores.head(top_n).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Global Mean vs Conditional Mean
axes[0].scatter(df_top['Conditional_Mean_ATP'], df_top['Global_Mean_ATP'], 
                alpha=0.6, s=df_top['Activation_Count']*2)
axes[0].set_xlabel('Conditional Mean ATP (活性化時のみ平均)', fontsize=12)
axes[0].set_ylabel('Global Mean ATP (全サンプル平均)', fontsize=12)
axes[0].set_title('Global Mean vs Conditional Mean\n(点のサイズ=活性化回数)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Plot 2: Activation Rate vs Global Mean
axes[1].scatter(df_top['Activation_Rate'], df_top['Global_Mean_ATP'], 
                alpha=0.6, c=df_top['Conditional_Mean_ATP'], cmap='viridis', s=100)
axes[1].set_xlabel('Activation Rate (活性化率)', fontsize=12)
axes[1].set_ylabel('Global Mean ATP', fontsize=12)
axes[1].set_title('頻度(Activation Rate) vs 効果(Global Mean)\n(色=Conditional Mean)', fontsize=14)
axes[1].grid(True, alpha=0.3)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Conditional Mean ATP')

plt.tight_layout()
plt.show()

print("レアケース（低頻度だが高スコア）の特徴を確認:")
print(df_atp_scores[df_atp_scores['Activation_Rate'] < 0.1].sort_values('Conditional_Mean_ATP', ascending=False).head(10))

## Base強度によるフィルタリング（ハイブリッド評価）

以前の `feature_metrics_full.csv` から `Mean Intensity Base` を取得し、
汎用的すぎる特徴（常に活性化している特徴）を除外します。

In [ ]:
# 結合するためにデータ型を変更
df_atp_scores["Feature_ID"] = df_atp_scores["Feature_ID"].astype(int)

In [ ]:
# data_metricsから必要な列を抽出
base_metrics = data_metrics[['Feature_ID', 'Mean Intensity Base']].copy()

# AtPスコアとマージ
df_combined = pd.merge(df_atp_scores, base_metrics, on='Feature_ID', how='right')

# Base強度でフィルタリング（閾値は要調整）
base_intensity_threshold = 0.5  # この値は後で調整可能

df_filtered = df_combined[
    (df_combined['Mean Intensity Base'] < base_intensity_threshold) | 
    (df_combined['Mean Intensity Base'].isna())
].copy()

print(f"フィルタリング前: {len(df_combined)} 特徴")
print(f"フィルタリング後: {len(df_filtered)} 特徴")
print(f"\n除外された特徴の例（Base強度が高い）:")
print(df_combined[df_combined['Mean Intensity Base'] >= base_intensity_threshold].sort_values('Global_Mean_ATP', ascending=False).head(10))

print(f"\n\n最終的な介入候補特徴（Top 20）:")
df_filtered_top = df_filtered.sort_values('Global_Mean_ATP', ascending=False).head(20)
df_filtered_top

## 評価指標の説明と結果の保存

### 最終評価式
$$\text{Score}(f_i) = \frac{1}{N_{syc}} \sum_{j=1}^{N_{syc}} \text{AtP}(f_i, x_j)$$

ここで:
- $N_{syc}$: 全迎合サンプル数（sycophancy_flag=1のサンプル数）
- $\text{AtP}(f_i, x_j)$: 特徴 $f_i$ がサンプル $x_j$ で活性化しなかった場合は0

### フィルタリング条件
- `Mean Intensity Base < 閾値`: 中立時に常時活性化している汎用特徴を除外

In [ ]:
# 結果をCSVファイルとして保存
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"atp_calculated_results/global_mean_atp_features_{timestamp}.csv"

# 全特徴の結果を保存
df_combined.sort_values('Global_Mean_ATP', ascending=False).to_csv(
    output_path.replace('.csv', '_all.csv'), 
    index=False
)

# フィルタリング後の結果を保存
df_filtered.sort_values('Global_Mean_ATP', ascending=False).to_csv(
    output_path.replace('.csv', '_filtered.csv'), 
    index=False
)

print(f"結果を保存しました:")
print(f"  全特徴: {output_path.replace('.csv', '_all.csv')}")
print(f"  フィルタ済: {output_path.replace('.csv', '_filtered.csv')}")